# Partial Freezing of a Multilingual Model for Chinese PoS Tagging

Compare full fine-tuning with freezing embeddings and the first three of DistilBERT’s six layers on a fixed Universal Dependencies Chinese GSD sample. The metric is word-level UPOS accuracy on the official dev split.

## 1. Install dependencies

Colab already provides PyTorch. This installs the Hugging Face model library and the small download dependency.

In [ ]:
%pip install -q 'transformers>=4.45,<5' requests

## 2. Get the code and check the runtime

If you opened the notebook from GitHub, this checks out the companion script. If you cloned the repository and opened the notebook inside it, it uses those files.

In [ ]:
from pathlib import Path
import os, platform, subprocess, torch
if not Path('run_experiment.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/billchenziang33/csc394-pos-partial-freezing.git'], check=True)
    os.chdir('csc394-pos-partial-freezing')
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert Path('run_experiment.py').exists()

## 3. Run paired comparisons across three seeds

For each seed, the wrapper runs both conditions on the same sampled sentences, with matched model initialization and training order. It saves each run and aggregates mean accuracy, sample standard deviation, and runtime.

In [ ]:
%run run_repeated_experiments.py --seeds 42 43 44

## 4. Read the measured scores

These are the results from the three-seed run committed in `results.json`. Rerun the cell above to reproduce them.

In [4]:
import json
from pathlib import Path
results = json.loads(Path('results.json').read_text(encoding='utf-8'))
for condition, row in results['summary'].items():
    sd = row['sample_stdev_dev_word_accuracy']
    spread = f' +/- {sd:.2%}' if sd is not None else ''
    print(f"{condition}: {row['mean_dev_word_accuracy']:.2%}{spread}; mean run time {row['mean_elapsed_seconds']:.1f}s")
print('Paired scores by seed:')
for run in results['runs']:
    scores = {row['condition']: row['dev_word_accuracy'] for row in run['results']}
    gap = scores['partial_freezing'] - scores['full_finetuning']
    print(f"seed {run['seed']}: full={scores['full_finetuning']:.2%}, frozen={scores['partial_freezing']:.2%}, frozen-minus-full={gap:+.2%}")

full_finetuning: 88.42% +/- 1.02%; mean run time 17.7s
partial_freezing: 86.56% +/- 1.03%; mean run time 9.4s
Paired scores by seed:
seed 42: full=87.55%, frozen=85.57%, frozen-minus-full=-1.97%
seed 43: full=89.55%, frozen=87.63%, frozen-minus-full=-1.91%
seed 44: full=88.18%, frozen=86.46%, frozen-minus-full=-1.72%


## 5. Interpretation

Across these three paired runs, full fine-tuning scored higher each time. Partial freezing trained about 15.8% of the parameters and ran faster in this small T4 experiment. This result is limited to one language, model, and sample size; the three seeds vary both the sampled sentences and random initialization, so they do not isolate those sources of variation.